# Simple base-rate merged results

Load multi-model results from downloaded Kaggle Benchmarks runs, or from a local merged CSV.

Each row has **`score`** (`true`/`false`). See `docs/benchmark-design-factors.md` for the simple benchmark: variants (`mc_prob`, `mc_w_meta`, `data_audit`, `response_audit`), conditions (`natural`, `altered`), and scoring keys. **`path_c_confusion`** flags answers matching **P(T|C)** (inverse-conditional lure) on numeric variants.

**Note:** Kaggle runs must use current `example_id`s (`{vignette}__{natural|altered}__{variant}`). Older runs are skipped at load time.

**Kaggle (all evaluated models):** after `kaggle auth login`:

```powershell
python -m kaggle benchmarks tasks download simple-rate-normative-accuracy `
  -o data/kaggle_runs/simple-rate-normative-accuracy
```

Or: `python scripts/export_simple_rate_kaggle_results.py --download`

Set `LOAD_FROM_KAGGLE = True` in the next cell (default).

In [3]:
from pathlib import Path
import importlib
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "simple").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload so notebook picks up merge/scoring changes without a full kernel restart.
import benchmarks.base_rate as base_rate
import benchmarks.kaggle_runs as kaggle_runs
import benchmarks.simple_rate as simple_rate

importlib.reload(base_rate)
importlib.reload(simple_rate)
importlib.reload(kaggle_runs)

from benchmarks.kaggle_runs import (
    DEFAULT_SIMPLE_RATE_TASK_SLUG,
    download_task_runs,
    merged_simple_results_from_kaggle_runs,
)

print("benchmarks.kaggle_runs:", kaggle_runs.__file__)

LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = False  # True to refresh via Kaggle CLI before loading
KAGGLE_TASK_SLUG = DEFAULT_SIMPLE_RATE_TASK_SLUG
KAGGLE_RUNS_DIR = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
BENCHMARK_CSV = ROOT / "data" / "simple" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "simple"

if LOAD_FROM_KAGGLE:
    if DOWNLOAD_KAGGLE_RUNS:
        download_task_runs(KAGGLE_TASK_SLUG, KAGGLE_RUNS_DIR)
    if not KAGGLE_RUNS_DIR.is_dir():
        raise FileNotFoundError(
            f"Download directory not found: {KAGGLE_RUNS_DIR}\n"
            f"Run: python -m kaggle benchmarks tasks download {KAGGLE_TASK_SLUG} "
            f"-o {KAGGLE_RUNS_DIR}"
        )
    # Filters stale example_ids; raises if no rows match current benchmark.csv.
    merged_rows = merged_simple_results_from_kaggle_runs(
        KAGGLE_RUNS_DIR,
        benchmark_path=BENCHMARK_CSV,
        fill_missing=False,
    )
    df = pd.DataFrame(merged_rows)
    data_source = f"Kaggle runs ({KAGGLE_RUNS_DIR})"
else:
    merged_candidates = sorted(
        MERGED_DIR.glob("simple_merged_results*.csv"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not merged_candidates:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/simple-benchmark.ipynb."
        )
    MERGED_CSV = merged_candidates[0]
    df = pd.read_csv(MERGED_CSV)
    data_source = str(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
else:
    raise KeyError("Merged data must include 'score'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")
else:
    df["parseable_bool"] = True

if "path_c_confusion" in df.columns:
    df["path_c_confusion_bool"] = (
        df["path_c_confusion"].astype(str).str.lower().eq("true")
    )
else:
    df["path_c_confusion_bool"] = False

df["score_true"] = df["score_value"].astype(bool)

VARIANT_ORDER = ["mc_prob", "mc_w_meta", "data_audit", "response_audit"]
CONDITION_ORDER = ["natural", "altered"]
INTERSECTION_SIZE_ORDER = ["0", "small", "medium", "large"]
PROBLEM_TYPE_ORDER = ["well_posed", "altered"]
SCEPTICISM_REQUIRED_ORDER = ["false", "true"]

print("Source:", data_source)
print("Rows:", len(df))
n_empty_loaded = int(
    (
        df["llm_response"].isna()
        | df["llm_response"].astype(str).str.strip().eq("")
    ).sum()
)
print("Empty llm_response at load:", n_empty_loaded, "/", len(df))
if LOAD_FROM_KAGGLE and n_empty_loaded:
    raise RuntimeError(
        "Padded empty rows detected — merge should use fill_missing=False. "
        "Check benchmarks/simple_rate.py and re-run this cell."
    )
if not LOAD_FROM_KAGGLE and n_empty_loaded:
    print(
        "Tip: stale padded CSV — set LOAD_FROM_KAGGLE=True or re-export with "
        "scripts/export_simple_rate_kaggle_results.py"
    )
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
print(
    "Normative pass:",
    int(df["score_true"].sum()),
    "/",
    len(df),
    "| P(T|C) confusion:",
    int(df["path_c_confusion_bool"].sum()),
    "/",
    len(df),
)
df.head()

benchmarks.kaggle_runs: c:\src2\sceptical-llms\benchmarks\kaggle_runs.py
Skipped 223 stale run row(s) with example_ids not in benchmark.csv.
Source: Kaggle runs (c:\src2\sceptical-llms\data\kaggle_runs\simple-rate-normative-accuracy)
Rows: 176
Empty llm_response at load: 0 / 176
Models: ['google/gemini-3-flash-preview']
Vignettes: 22
Normative pass: 148 / 176 | P(T|C) confusion: 3 / 176


,example_id,vignette_name,condition,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,...,parsed_confidence,comment_line,scoring_type,parseable,score,path_c_confusion,score_value,parseable_bool,path_c_confusion_bool,score_true
0,book_drama_streaming__altered__data_audit,book drama streaming,altered,altered,large,data_audit,true,data_audit,You are a data analyst. Your customer has aske...,false,...,,,mc_numeric,true,true,false,1,True,False,True
1,book_drama_streaming__altered__mc_prob,book drama streaming,altered,altered,large,mc,true,mc_prob,You are a statistical consultant. Your task is...,false,...,,,mc_numeric,true,true,false,1,True,False,True
2,book_drama_streaming__altered__mc_w_meta,book drama streaming,altered,altered,large,mc_full,true,mc_w_meta,You are a statistical consultant. Your task is...,false,...,,,mc_full,true,false,false,0,True,False,False
3,book_drama_streaming__altered__response_audit,book drama streaming,altered,altered,large,response_audit,true,response_audit,"You are a data auditor, checking the work of a...",false,...,,,mc_numeric,true,false,false,0,True,False,False
4,book_drama_streaming__natural__data_audit,book drama streaming,natural,well_posed,large,data_audit,true,data_audit,You are a data analyst. Your customer has aske...,false,...,,,mc_numeric,true,false,false,0,True,False,False


In [4]:
df.columns


Index(['example_id', 'vignette_name', 'condition', 'problem_type',
       'intersection_size', 'response_type', 'has_statistics', 'variant',
       'prompt', 'well_posed', 'normative', 'p_c_and_d_given_a', 'p_c', 'p_d',
       'p_t_given_c', 'p_t_given_d', 'normative_choice', 'normative_percent',
       'normative_open', 'confidence_required', 'numeric_score_percent',
       'numeric_score_choice', 'scepticism_required',
       'scepticism_score_target', 'option_a_label', 'option_b_label',
       'option_c_label', 'option_d_label', 'option_e_label', 'option_a_lure',
       'option_b_lure', 'option_c_lure', 'option_d_lure', 'option_e_lure',
       'option_f_label', 'option_g_label', 'option_h_label', 'option_f_lure',
       'option_g_lure', 'option_h_lure', 'model', 'llm_response', 'reasoning',
       'answer_line', 'confidence_line', 'parsed_answer_type',
       'parsed_percent', 'parsed_choice', 'parsed_confidence', 'comment_line',
       'scoring_type', 'parseable', 'score', 'path_c_

## Model ID reference (Anthropic)

Kaggle results use provider-prefixed IDs like `anthropic/claude-haiku-4-5@20251001`. For pre–4.6 Claude models, the part after `@` is a **snapshot date** (`YYYYMMDD`) — a pinned release, not “latest.” `@default` is a platform alias (not a date) that routes to the default endpoint for that model line.

Naming: `claude-{tier}-{major}-{minor}` → e.g. **Haiku 4.5** = fast tier, generation 4, minor version 5. See [Anthropic model IDs](https://platform.claude.com/docs/en/about-claude/models/model-ids-and-versions).

In [5]:
ANTHROPIC_MODEL_REFERENCE = pd.DataFrame(
    [
        {
            "model_id": "anthropic/claude-haiku-4-5@20251001",
            "marketing_name": "Claude Haiku 4.5",
            "tier": "Haiku",
            "snapshot": "2025-10-01",
            "in_results": False,
            "notes": "Fastest/cheapest tier; near-frontier on coding/agents. Vertex-style @ date.",
        },
        {
            "model_id": "anthropic/claude-sonnet-4@20250514",
            "marketing_name": "Claude Sonnet 4",
            "tier": "Sonnet",
            "snapshot": "2025-05-14",
            "in_results": False,
            "notes": "Balanced speed/intelligence; Sonnet 4 (pre-4.5) snapshot.",
        },
        {
            "model_id": "anthropic/claude-opus-4-1@20250805",
            "marketing_name": "Claude Opus 4.1",
            "tier": "Opus",
            "snapshot": "2025-08-05",
            "in_results": False,
            "notes": "Legacy flagship; Anthropic deprecated (retirement Aug 2026).",
        },
        {
            "model_id": "anthropic/claude-opus-4-8@default",
            "marketing_name": "Claude Opus 4.8",
            "tier": "Opus",
            "snapshot": "default",
            "in_results": False,
            "notes": "Current flagship; @default = platform alias, not a calendar date.",
        },
    ]
)

_models_in_df = set(df["model"].unique())
ANTHROPIC_MODEL_REFERENCE["in_results"] = ANTHROPIC_MODEL_REFERENCE["model_id"].isin(
    _models_in_df
)

anthropic_in_results = ANTHROPIC_MODEL_REFERENCE.loc[
    ANTHROPIC_MODEL_REFERENCE["in_results"]
].reset_index(drop=True)
other_models = sorted(_models_in_df - set(ANTHROPIC_MODEL_REFERENCE["model_id"]))

print("Anthropic models in this merge:")
display(
    anthropic_in_results[
        ["model_id", "marketing_name", "tier", "snapshot", "notes"]
    ]
)

if other_models:
    print("Other providers in this merge:")
    for model_id in other_models:
        print(f"  {model_id}")

unknown_anthropic = sorted(
    model_id
    for model_id in _models_in_df
    if model_id.startswith("anthropic/")
    and model_id not in set(ANTHROPIC_MODEL_REFERENCE["model_id"])
)
if unknown_anthropic:
    print("Anthropic IDs not in reference table (add rows above):")
    for model_id in unknown_anthropic:
        print(f"  {model_id}")

Anthropic models in this merge:


,model_id,marketing_name,tier,snapshot,notes


Other providers in this merge:
  google/gemini-3-flash-preview


In [6]:
from IPython.display import Markdown, display

df["empty_llm_response_bool"] = (
    df["llm_response"].isna()
    | df["llm_response"].astype(str).str.strip().eq("")
)
n_empty = int(df["empty_llm_response_bool"].sum())
print(
    f"Empty llm_response: {n_empty} / {len(df)} "
    f"({df['empty_llm_response_bool'].mean() * 100:.1f}%)"
)

if n_empty > 0:

    def empty_response_summary_table(
        group_col: str, *, order: list[str] | None = None
    ) -> pd.DataFrame:
        work = df.copy()
        if group_col == "scepticism_required":
            work[group_col] = work[group_col].astype(str).str.lower()
        grouped = work.groupby(group_col, observed=True)
        summary = pd.DataFrame(
            {
                "n": grouped.size(),
                "empty_llm_response": grouped["empty_llm_response_bool"].sum().astype(int),
            }
        )
        summary["empty_pct"] = (summary["empty_llm_response"] / summary["n"] * 100).round(1)
        if order is not None:
            summary = summary.reindex([value for value in order if value in summary.index])
        elif group_col == "model":
            summary = summary.sort_index()
        return summary

    for title, col, order in [
        ("variant", "variant", VARIANT_ORDER),
        ("problem_type", "problem_type", PROBLEM_TYPE_ORDER),
        ("scepticism_required", "scepticism_required", SCEPTICISM_REQUIRED_ORDER),
        ("intersection_size", "intersection_size", INTERSECTION_SIZE_ORDER),
        ("model", "model", None),
        ("vignette", "vignette_name", sorted(df["vignette_name"].unique())),
    ]:
        display(Markdown(f"### By {title}"))
        display(empty_response_summary_table(col, order=order))

Rows with empty llm_response: 0 / 176


In [8]:
def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, normative score, and P(T|C) confusion by group."""
    work = df.copy()
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "path_c_confusion": grouped["path_c_confusion_bool"].sum(),
        }
    )
    summary["score_pct"] = (grouped["score_value"].mean() * 100).round(1)
    summary["path_c_pct"] = (grouped["path_c_confusion_bool"].mean() * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])
    return summary


score_summary_table("variant", order=VARIANT_ORDER)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
variant,,,,,,,
mc_prob,44,33,11,0,3,75.0,6.8
mc_w_meta,44,38,6,0,0,86.4,0.0
data_audit,44,41,3,0,0,93.2,0.0
response_audit,44,36,8,0,0,81.8,0.0


## By problem_type

In [9]:
score_summary_table(
    "problem_type",
    order=[value for value in PROBLEM_TYPE_ORDER if value in df["problem_type"].unique()],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
problem_type,,,,,,,
well_posed,88,74,14,0,1,84.1,1.1
altered,88,74,14,0,2,84.1,2.3


## By scepticism_required

In [10]:
score_summary_table(
    "scepticism_required",
    order=[
        value
        for value in SCEPTICISM_REQUIRED_ORDER
        if value in df["scepticism_required"].astype(str).str.lower().unique()
    ],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
scepticism_required,,,,,,,
false,77,64,13,0,3,83.1,3.9
true,99,84,15,0,0,84.8,0.0


## By intersection_size

In [11]:
score_summary_table(
    "intersection_size",
    order=[value for value in INTERSECTION_SIZE_ORDER if value in df["intersection_size"].unique()],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
intersection_size,,,,,,,
0,88,79,9,0,1,89.8,1.1
large,88,69,19,0,2,78.4,2.3


In [12]:
score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
vignette_name,,,,,,,
CA Trump voter,8,7,1,0,1,87.5,12.5
HS graduation ACGR (WV vs AZ),8,7,1,0,0,87.5,0.0
NAEP grade 4 reading (MA vs NM),8,8,0,0,0,100.0,0.0
NFL MLB watch attend,8,5,3,0,0,62.5,0.0
book drama streaming,8,5,3,0,0,62.5,0.0
college grad professional job,8,8,0,0,0,100.0,0.0
covid vaccine (blue/red),8,8,0,0,0,100.0,0.0
diabetes insulin obese,8,8,0,0,0,100.0,0.0
discharged weapon (last year),8,7,1,0,0,87.5,0.0


## By model

In [13]:
score_summary_table(
    "model",
    order=sorted(df["model"].unique()),
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
model,,,,,,,
google/gemini-3-flash-preview,176,148,28,0,3,84.1,1.7


## By model × variant

Mean normative **score_pct** and row count **n** per model and variant. Full coverage is **9** for `open_probs` / `mc_numeric_probs` and **27** for `mc_full_probs` (45 total per model).

In [14]:
def model_variant_score_table() -> pd.DataFrame:
    """Pivot: rows=model, columns=(metric, variant) with score_pct and n."""
    grouped = df.groupby(["model", "variant"], observed=True)["score_value"]
    flat = grouped.agg(
        n="size",
        score_pct=lambda values: round(values.mean() * 100, 1),
    )
    score_pct = flat["score_pct"].unstack("variant").reindex(columns=VARIANT_ORDER)
    counts = flat["n"].unstack("variant").reindex(columns=VARIANT_ORDER)
    return pd.concat({"score_pct": score_pct, "n": counts}, axis=1).sort_index()


model_variant_score_table()

score_pct                                      \
variant                         mc_prob mc_w_meta data_audit response_audit   
model                                                                         
google/gemini-3-flash-preview      75.0      86.4       93.2           81.8   

                                    n                                      
variant                       mc_prob mc_w_meta data_audit response_audit  
model                                                                      
google/gemini-3-flash-preview      44        44         44             44

## Haiku: `mc_numeric_probs` vs `mc_full_probs`

`mc_numeric_probs` is **well-posed only** (9 vignettes, no F/G/H). `mc_full_probs` adds meta options and **18 scepticism-required** implausible forks — so a lower headline score often mixes (1) picking F/G on overlap vignettes when only A–E are offered in numeric MC, and (2) failing to pick **H** on implausible items.

Per vignette below: source probabilities, `scepticism_required`, parsed letter, score target, and full prompt for each MC variant present in the merged runs.

In [15]:
HAIKU_MODEL = "anthropic/claude-haiku-4-5@20251001"

haiku = df.loc[df["model"] == HAIKU_MODEL].copy()
if haiku.empty:
    raise ValueError(f"No rows for {HAIKU_MODEL!r}. Models: {sorted(df['model'].unique())}")

MC_VARIANTS = ("mc_numeric_probs", "mc_full_probs")
mc = haiku.loc[haiku["variant"].isin(MC_VARIANTS)].copy()
mc["scepticism_required_bool"] = (
    mc["scepticism_required"].astype(str).str.lower().eq("true")
)
mc["score_target"] = mc.apply(
    lambda row: row["scepticism_score_target"]
    if row["scepticism_required_bool"]
    else row["normative_choice"],
    axis=1,
)

print(f"Model: {HAIKU_MODEL}")
print(f"MC rows: {len(mc)}")
print()
print("Score by variant (all problem types):")
display(
    mc.groupby("variant", observed=True)["score_value"]
    .agg(score_pct=lambda s: round(s.mean() * 100, 1), n="size", score_true="sum")
    .reindex(MC_VARIANTS)
)
print("Well-posed only (apples-to-apples normative P(C|T)):")
display(
    mc.loc[mc["problem_type"] == "well_posed"]
    .groupby("variant", observed=True)["score_value"]
    .agg(score_pct=lambda s: round(s.mean() * 100, 1), n="size", score_true="sum")
    .reindex(MC_VARIANTS)
)
print("mc_full_probs by problem_type:")
display(
    mc.loc[mc["variant"] == "mc_full_probs"]
    .groupby("problem_type", observed=True)["score_value"]
    .agg(score_pct=lambda s: round(s.mean() * 100, 1), n="size", score_true="sum")
    .reindex(PROBLEM_TYPE_ORDER)
)

detail_cols = [
    "vignette_name",
    "variant",
    "problem_type",
    "scepticism_required",
    "p_c",
    "p_d",
    "p_t_given_c",
    "p_t_given_d",
    "parsed_choice",
    "score_target",
    "score",
    "path_c_confusion",
]
haiku_mc_detail = (
    mc[detail_cols]
    .sort_values(["vignette_name", "variant", "problem_type"])
    .reset_index(drop=True)
)
print("Per-row detail:")
display(haiku_mc_detail)

prob_cols = ["p_c", "p_d", "p_t_given_c", "p_t_given_d"]


def _prob_summary(row: pd.Series) -> str:
    return (
        f"P(C)={float(row['p_c']):.4g}  P(D)={float(row['p_d']):.4g}  "
        f"P(T|C)={float(row['p_t_given_c']):.4g}  P(T|D)={float(row['p_t_given_d']):.4g}"
    )


for vignette_name in sorted(mc["vignette_name"].unique()):
    vignette_rows = mc.loc[mc["vignette_name"] == vignette_name].sort_values(
        ["variant", "problem_type"]
    )
    print("=" * 80)
    print(vignette_name)
    print("=" * 80)

    for variant in MC_VARIANTS:
        variant_rows = vignette_rows.loc[vignette_rows["variant"] == variant]
        if variant_rows.empty:
            print(f"\n[{variant}] — no run in merged data")
            continue
        for _, row in variant_rows.iterrows():
            scept = row["scepticism_required_bool"]
            print(
                f"\n[{variant}]  problem_type={row['problem_type']}  "
                f"scepticism_required={row['scepticism_required']}"
            )
            print(f"  probs: {_prob_summary(row)}")
            print(
                f"  parsed={row['parsed_choice']!r}  target={row['score_target']!r}  "
                f"score={row['score']}  path_c_confusion={row['path_c_confusion']}"
            )
            print("  prompt:")
            for line in str(row["prompt"]).strip().splitlines():
                print(f"    {line}")
    print()

ValueError: No rows for 'anthropic/claude-haiku-4-5@20251001'. Models: ['google/gemini-3-flash-preview']

In [ ]:
# Haiku: side-by-side MC answers where both variants exist for the same vignette.
_haiku_model = "anthropic/claude-haiku-4-5@20251001"
_haiku = df.loc[df["model"] == _haiku_model]
_mc = _haiku.loc[_haiku["variant"].isin(["mc_numeric_probs", "mc_full_probs"])]

_numeric_vignettes = set(
    _mc.loc[_mc["variant"] == "mc_numeric_probs", "vignette_name"]
)
_full_vignettes = set(_mc.loc[_mc["variant"] == "mc_full_probs", "vignette_name"])
_paired_vignettes = sorted(_numeric_vignettes & _full_vignettes)

print(
    f"{_haiku_model}: {len(_paired_vignettes)} vignettes with both MC variants "
    f"(numeric-only: {len(_numeric_vignettes - _full_vignettes)})"
)

paired_rows: list[dict[str, object]] = []
for vignette_name in _paired_vignettes:
    numeric = _mc.loc[
        (_mc["vignette_name"] == vignette_name)
        & (_mc["variant"] == "mc_numeric_probs")
    ].iloc[0]
    full_by_type = {
        row["problem_type"]: row
        for _, row in _mc.loc[
            (_mc["vignette_name"] == vignette_name)
            & (_mc["variant"] == "mc_full_probs")
        ].iterrows()
    }

    entry: dict[str, object] = {
        "vignette_name": vignette_name,
        "mc_numeric_choice": numeric["parsed_choice"],
        "mc_numeric_target": numeric["normative_choice"],
        "mc_numeric_score": numeric["score"],
        "mc_numeric_scepticism_required": numeric["scepticism_required"],
    }
    for problem_type in PROBLEM_TYPE_ORDER:
        prefix = f"mc_full_{problem_type}"
        if problem_type in full_by_type:
            full = full_by_type[problem_type]
            scepticism = str(full["scepticism_required"]).lower() == "true"
            entry[f"{prefix}_choice"] = full["parsed_choice"]
            entry[f"{prefix}_target"] = (
                full["scepticism_score_target"]
                if scepticism
                else full["normative_choice"]
            )
            entry[f"{prefix}_score"] = full["score"]
            entry[f"{prefix}_scepticism_required"] = full["scepticism_required"]
        else:
            entry[f"{prefix}_choice"] = pd.NA
            entry[f"{prefix}_target"] = pd.NA
            entry[f"{prefix}_score"] = pd.NA
            entry[f"{prefix}_scepticism_required"] = pd.NA
    paired_rows.append(entry)

haiku_paired_mc = pd.DataFrame(paired_rows)

display_cols = [
    "vignette_name",
    "mc_numeric_choice",
    "mc_numeric_target",
    "mc_numeric_score",
    "mc_numeric_scepticism_required",
    "mc_full_well_posed_choice",
    "mc_full_well_posed_target",
    "mc_full_well_posed_score",
    "mc_full_well_posed_scepticism_required",
    "mc_full_implausible_c_d_choice",
    "mc_full_implausible_c_d_target",
    "mc_full_implausible_c_d_score",
    "mc_full_implausible_c_d_scepticism_required",
    "mc_full_implausible_t_choice",
    "mc_full_implausible_t_target",
    "mc_full_implausible_t_score",
    "mc_full_implausible_t_scepticism_required",
]
haiku_paired_mc[display_cols]

## Unparseable responses

Model and raw response for rows where the answer could not be parsed.

In [ ]:
unparseable = df.loc[~df["parseable_bool"]].sort_values(["model", "example_id"])
print(f"{len(unparseable)} unparseable rows")

pd.set_option("display.max_colwidth", None)
unparseable[["model", "llm_response"]]

## `mc_numeric_probs` detail

MC options, parsed letter, normative letter, score, and whether the choice is the **P(T|C)** lure.

In [ ]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]
MC_LURE_COLS = [f"option_{letter}_lure" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, label_col, lure_col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS, MC_LURE_COLS):
        label = row.get(label_col)
        lure = row.get(lure_col)
        if pd.notna(label) and str(label).strip():
            lure_text = f" [{lure}]" if pd.notna(lure) and str(lure).strip() else ""
            parts.append(f"{letter}: {label}{lure_text}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "normative_choice",
        "p_t_given_c",
        "score",
        "score_value",
        "path_c_confusion",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 160)
mc_numeric_probs_view

## `open_probs` detail

Re-parse open responses and compare to normative **P(C|T)** and **P(T|C)**.

In [ ]:
from benchmarks.base_rate import matches_scepticism_target, parse_open_response
from benchmarks.simple_rate import PATH_C_LURE_NAME, load_benchmark, matches_path_c_confusion

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    return pd.Series(
        {
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.answer_type != "unparseable",
            "score_true": matches_scepticism_target(item, parsed),
            "path_c_confusion_rescored": matches_path_c_confusion(item, parsed),
            "p_t_given_c_pct": float(row["p_t_given_c"]) * 100,
        }
    )


def _csv_bool(series: pd.Series) -> pd.Series:
    return series.astype(bool).map(lambda value: "true" if value else "false")


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = open_probs.drop(columns=["score_true"], errors="ignore")
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

idx = open_probs.index
rescored_percent = pd.to_numeric(open_probs["parsed_percent_rescored"], errors="coerce")
df.loc[idx, "parsed_percent"] = rescored_percent.map(
    lambda value: "" if pd.isna(value) else f"{value:g}"
)
df.loc[idx, "score_true"] = open_probs["score_true"].astype(bool)
df.loc[idx, "score_value"] = open_probs["score_true"].astype(int)
if "score" in df.columns:
    df.loc[idx, "score"] = _csv_bool(open_probs["score_true"])
df.loc[idx, "path_c_confusion_bool"] = open_probs["path_c_confusion_rescored"].astype(bool)
if "path_c_confusion" in df.columns:
    df.loc[idx, "path_c_confusion"] = _csv_bool(open_probs["path_c_confusion_rescored"])

open_probs_view = open_probs[
    [
        "example_id",
        "vignette_name",
        "normative_percent",
        "normative_open",
        "p_t_given_c_pct",
        "parsed_numbers",
        "parsed_percent_rescored",
        "score_true",
        "path_c_confusion_rescored",
    ]
].sort_values("vignette_name")

print(
    "Rescored open_probs normative pass:",
    int(open_probs["score_true"].sum()),
    "/",
    len(open_probs),
    "| P(T|C) confusion:",
    int(open_probs["path_c_confusion_rescored"].sum()),
    "/",
    len(open_probs),
)
open_probs_view

## `open_probs` vs `mc_numeric_probs` vs normative

Side-by-side for all 10 vignettes. **Normative** = P(C|T) (`normative_percent`). **P(T|C)** = `p_t_given_c` (inverse-conditional lure).

In [ ]:
from benchmarks.base_rate import parse_open_response, parse_response
from benchmarks.simple_rate import PATH_C_LURE_NAME
from scripts.build_base_rate_prompts import _load_overlap

OVERLAP_VIGNETTE_NAMES = {v.name for v in _load_overlap()}

items_meta = pd.read_csv(ROOT / "data" / "simple" / "items.csv")


def _label_percent(label: str) -> float | None:
    text = (label or "").strip()
    if not text.startswith("About "):
        return None
    try:
        return float(text.removeprefix("About ").removesuffix("%"))
    except ValueError:
        return None


def _format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter in "ABCDE":
        label = row.get(f"option_{letter.lower()}_label")
        if pd.notna(label) and str(label).strip():
            parts.append(f"{letter}: {label}")
    return " | ".join(parts)


comparison_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = item_mc.get(f"option_{mc_choice.lower()}_label", "") if mc_choice else ""
    mc_lure = item_mc.get(f"option_{mc_choice.lower()}_lure", "") if mc_choice else ""
    mc_pct = _label_percent(str(mc_label))

    normative_pct = float(item_open["normative_percent"])
    path_c_pct = float(item_open["p_t_given_c"]) * 100
    open_pct = parsed_open.percent

    comparison_rows.append(
        {
            "vignette_name": vignette_name,
            "overlap": vignette_name in OVERLAP_VIGNETTE_NAMES,
            "normative_open": item_open["normative_open"],
            "normative_pct": normative_pct,
            "p_t_given_c_pct": path_c_pct,
            "open_parsed_pct": open_pct,
            "open_delta_vs_norm_pp": None if open_pct is None else open_pct - normative_pct,
            "open_score": bool(open_row.get("score_true", open_row.get("score_value", 0))),
            "open_path_c": bool(open_row.get("path_c_confusion_bool", False)),
            "mc_choices": _format_mc_choices(item_mc),
            "mc_choice": mc_choice,
            "mc_label": mc_label,
            "mc_lure": mc_lure,
            "mc_parsed_pct": mc_pct,
            "mc_delta_vs_norm_pp": None if mc_pct is None else mc_pct - normative_pct,
            "mc_score": bool(mc_row.get("score_true", mc_row.get("score_value", 0))),
            "mc_path_c": mc_lure == PATH_C_LURE_NAME or (
                mc_pct is not None and abs(mc_pct - path_c_pct) <= 0.5
            ),
            "normative_mc_letter": item_mc["normative_choice"],
        }
    )

open_vs_mc = pd.DataFrame(comparison_rows).sort_values("vignette_name")

print(
    "open_probs pass:",
    int(open_vs_mc["open_score"].sum()),
    "/",
    len(open_vs_mc),
    "| mc_numeric_probs pass:",
    int(open_vs_mc["mc_score"].sum()),
    "/",
    len(open_vs_mc),
    "| open P(T|C) confusion:",
    int(open_vs_mc["open_path_c"].sum()),
    "| mc P(T|C) confusion:",
    int(open_vs_mc["mc_path_c"].sum()),
)

COMPARISON_COLUMNS = [
    "vignette_name",
    "overlap",
    "normative_pct",
    "open_parsed_pct",
    "open_score",
    "mc_choices",
    "mc_choice",
    "mc_label",
    "mc_score",
]

pd.set_option("display.max_colwidth", 160)
display(open_vs_mc[COMPARISON_COLUMNS])

basically, whenever overlap is false, everything is correct; but when overlap is true, the open output is wrong, but the mc is correct.
COuld this be due to the MC being way to easy?

### Printable comparison

Per vignette: source probabilities from `items.csv` (P(C), P(D), P(T|C), P(T|D)), normative / open / MC answers, and full `mc_numeric_probs` prompt.

In [ ]:
import re

from benchmarks.base_rate import parse_open_response, parse_response

benchmark_df = pd.read_csv(ROOT / "data" / "simple" / "benchmark.csv")


def mc_numeric_options_prompt(prompt: str) -> str:
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    option_lines = [line for line in lines if re.match(r"^[A-E]\.\s", line)]
    return " | ".join(option_lines)


def format_source_ps(item_row: pd.Series) -> str:
    return " | ".join(
        [
            f"P(C)={float(item_row['p_c']):.6g}",
            f"P(D)={float(item_row['p_d']):.6g}",
            f"P(T|C)={float(item_row['p_t_given_c']):.6g}",
            f"P(T|D)={float(item_row['p_t_given_d']):.6g}",
        ]
    )


print_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]
    bench_row = benchmark_df.loc[benchmark_df["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = (
        str(item_mc.get(f"option_{mc_choice.lower()}_label", ""))
        if mc_choice
        else ""
    )
    full_prompt = str(bench_row["prompt"])

    print_rows.append(
        {
            "vignette_name": vignette_name,
            "source_ps": format_source_ps(item_open),
            "normative_pct": float(item_open["normative_percent"]),
            "p_t_given_c_pct": float(item_open["p_t_given_c"]) * 100,
            "open_parsed_pct": parsed_open.percent,
            "mc_label": f"{mc_choice} {mc_label}".strip(),
            "numeric_prompt": mc_numeric_options_prompt(full_prompt),
            "prompt": full_prompt,
        }
    )

print_table = pd.DataFrame(print_rows).sort_values("vignette_name")

print(f"{'vignette_name':<32} {'normative':>10} {'P(T|C)':>10} {'open':>10} {'MC label':>14}")
print("-" * 84)
for row in print_table.itertuples(index=False):
    open_pct = "—" if pd.isna(row.open_parsed_pct) else f"{row.open_parsed_pct:.4g}%"
    print(f"\n{row.vignette_name}")
    print(f"  source Ps:   {row.source_ps}")
    print(f"  normative:   {row.normative_pct:.4g}%  (P(C|T))")
    print(f"  P(T|C):      {row.p_t_given_c_pct:.4g}%  (inverse-conditional lure)")
    print(f"  open parsed: {open_pct}")
    print(f"  MC label:    {row.mc_label}")
    print(f"  numeric prompt: {row.numeric_prompt}")
    print("  prompt:")
    for line in row.prompt.splitlines():
        print(f"    {line}")

print_table.drop(columns=["prompt"])

## Optional: split by model when multiple LLMs are present

In [ ]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["path_c_confusion_bool"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")